In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_11.pickle

In [ ]:
%%cudf.pandas.profile
### cell 11 ###

country_order = df['first_country'].value_counts()[:11].index

# Use GPU‐friendly pivot instead of CPU‐bound value_counts + unstack
counts = df.groupby(['first_country', 'type']).size().reset_index(name='count')
data_q2q3 = (
    counts
    .pivot(index='first_country', columns='type', values='count')
    .fillna(0)
    .loc[country_order]
)

# compute total and ratios
data_q2q3['sum'] = data_q2q3.sum(axis=1)
data_q2q3_ratio = (
    data_q2q3[['Movie', 'TV Show']]
    .div(data_q2q3['sum'], axis=0)
    .sort_values(by='Movie', ascending=False)
    .iloc[::-1]
)